In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler, SplineTransformer
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
startups = pd.read_csv("Startups.csv")

startups.head()

In [ ]:
X = startups[["RD"]]
y = startups["Profit"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=6021
)

In [ ]:
px.scatter(x=X_train["RD"], y=y_train,
           labels={"x": "R&D spending ($)", "y": "Profit ($)"},
           title="Startups: training data").show()

In [ ]:
X_train.describe()

In [ ]:
lin_model = Pipeline([
    ("scale_x", StandardScaler()),
    ("linreg", LinearRegression())
])

lin_model.fit(X_train, y_train)

In [ ]:
poly3_model = Pipeline([
    ("scale_x", StandardScaler()),
    ("poly", PolynomialFeatures(degree=3, include_bias=False)),
    ("linreg", LinearRegression())
])

poly3_model.fit(X_train, y_train)

In [ ]:
poly8_model = Pipeline([
    ("scale_x", StandardScaler()),
    ("poly", PolynomialFeatures(degree=8, include_bias=False)),
    ("scaler", StandardScaler()),
    ("linreg", LinearRegression())
])
poly8_model.fit(X_train, y_train)

In [ ]:
spline_model = Pipeline([
    ("spline", SplineTransformer(degree=3, n_knots=8, include_bias=False)),
    ("scaler", StandardScaler()),
    ("linreg", LinearRegression())
])
spline_model.fit(X_train, y_train)

In [ ]:
grid = pd.DataFrame({
    "RD": np.linspace(X_train["RD"].min(), X_train["RD"].max(), 200)
})

grid["Linear"] = lin_model.predict(grid[["RD"]])
grid["Poly 3"] = poly3_model.predict(grid[["RD"]])
grid["Poly 8"] = poly8_model.predict(grid[["RD"]])
grid["Spline"] = spline_model.predict(grid[["RD"]])

grid.head()

In [ ]:
fig = px.scatter(x=X_train["RD"], y=y_train,
                 labels={"x": "R&D spending ($)", "y": "Profit ($)"})

fig.add_trace(go.Scatter(x=grid["RD"], y=grid["Linear"], name="Linear"))
fig.add_trace(go.Scatter(x=grid["RD"], y=grid["Poly 3"], name="Poly 3"))
fig.add_trace(go.Scatter(x=grid["RD"], y=grid["Poly 8"], name="Poly 8"))
fig.add_trace(go.Scatter(x=grid["RD"], y=grid["Spline"], name="Spline"))

fig.update_layout(title="Same training data, four fitted models")
fig.show()

In [ ]:
fig = make_subplots(rows=2, cols=2,
                    subplot_titles=["Linear", "Poly 3", "Poly 8", "Spline"],
                    shared_xaxes=True, shared_yaxes=True)

for name, row, col in [("Linear", 1, 1), ("Poly 3", 1, 2),
                       ("Poly 8", 2, 1), ("Spline", 2, 2)]:
    fig.add_trace(go.Scatter(x=X_train["RD"], y=y_train,
                            mode="markers", marker_color="#58789D",
                            showlegend=False), row=row, col=col)
    fig.add_trace(go.Scatter(x=grid["RD"], y=grid[name],
                            mode="lines", line_color="#D97732",
                            showlegend=False), row=row, col=col)

fig.update_xaxes(title_text="R&D spending ($)")
fig.update_yaxes(title_text="Profit ($)")
fig.update_layout(title="Model flexibility: each fit separately", height=750)
fig.show()

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

models = {
    "Linear": lin_model,
    "Poly 3": poly3_model,
    "Poly 8": poly8_model,
    "Spline": spline_model
}

rows = []

for name, model in models.items():
    y_hat = model.predict(X_test)
    test_rmse = np.sqrt(mean_squared_error(y_test, y_hat))
    test_r2 = r2_score(y_test, y_hat)

    rows.append({
        "Model": name,
        "Test RMSE": test_rmse,
        "Test R²": test_r2
    })

results = pd.DataFrame(rows)
results.round(2)

In [ ]:
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=list(models.keys())
)

predictions = {
    name: model.predict(X_test)
    for name, model in models.items()
}

all_values = np.concatenate([
    np.asarray(y_test).ravel(),
    *[np.asarray(y_hat).ravel() for y_hat in predictions.values()]
])
low, high = all_values.min(), all_values.max()
padding = 0.05 * (high - low)
limits = [low - padding, high + padding]

for i, (name, y_hat) in enumerate(predictions.items()):
    row = i // 2 + 1
    col = i % 2 + 1

    fig.add_trace(
        go.Scatter(
            x=np.asarray(y_test).ravel(),
            y=np.asarray(y_hat).ravel(),
            mode="markers",
            marker=dict(size=8, opacity=0.7),
            name=name
        ),
        row=row, col=col
    )


    fig.add_trace(
        go.Scatter(
            x=limits, y=limits,
            mode="lines",
            line=dict(color="black", dash="dash"),
            showlegend=False,
            hoverinfo="skip"
        ),
        row=row, col=col
    )

fig.update_xaxes(title_text="Observed", range=limits)
fig.update_yaxes(title_text="Predicted", range=limits)

fig.update_layout(
    title="Predicted vs. Observed: Test Set",
    template="plotly_white",
    width=850,
    height=800,
    showlegend=False
)

fig.show()

In [ ]:
from sklearn.base import clone

X_new = pd.DataFrame({"RD": [150000]})

models = {
    "Linear": lin_model,
    "Poly 3": poly3_model,
    "Poly 8": poly8_model,
    "Spline": spline_model
}

B = 1000
rng = np.random.default_rng(123)
y_train_array = np.asarray(y_train).ravel()

rows = []

for name, model in models.items():


    y_hat = model.predict(X_new).item()


    residuals = y_train_array - model.predict(X_train).ravel()
    residuals = residuals - residuals.mean()

    mean_predictions = []
    individual_predictions = []

    for b in range(B):


        idx = rng.choice(
            len(X_train), size=len(X_train), replace=True
        )

        boot_model = clone(model)
        boot_model.fit(X_train.iloc[idx], y_train_array[idx])

        boot_prediction = boot_model.predict(X_new).item()


        mean_predictions.append(boot_prediction)


        random_error = rng.choice(residuals)
        individual_predictions.append(boot_prediction + random_error)

    ci_lower, ci_upper = np.percentile(
        mean_predictions, [2.5, 97.5]
    )

    pi_lower, pi_upper = np.percentile(
        individual_predictions, [2.5, 97.5]
    )

    rows.append({
        "Model": name,
        "Predicted Profit": y_hat,
        "95% CI Lower": ci_lower,
        "95% CI Upper": ci_upper,
        "95% PI Lower": pi_lower,
        "95% PI Upper": pi_upper
    })

prediction_results = pd.DataFrame(rows)
prediction_results.round(2)